# LLM 오분류 피드백 텍스트 추출 및 분석

## 목적
이 노트북은 LLM이 오분류한 피드백의 **실제 텍스트**를 추출하여, 어떤 유형의 피드백이 분류하기 어려운지 질적으로 분석합니다. 02_confusion_analysis에서 발견된 혼동 패턴의 원인을 텍스트 수준에서 규명합니다.

## 주요 분석
1. **모델별/프롬프트별 오분류율** 비교
2. **레이블별 오분류율**: 어떤 피드백 유형이 가장 분류하기 어려운가
3. **혼동 패턴별 텍스트 예시**: 왜 혼동이 발생하는지 실제 텍스트로 확인
4. **일관되게 오분류되는 고난이도 샘플**: 모든 조합에서 틀리는 "경계 사례(borderline case)"
5. **특정 조합에서만 발생하는 고유 오분류**: 모델/프롬프트의 약점 파악

## 레이블 정의
- **Label 0**: 답안과 무관한 내용으로 구성된 피드백
- **Label 1**: 글 전반에 대한 막연한 인상이나 소감만을 밝히며, 구체적인 평가 근거를 제시하지 않은 피드백
- **Label 2**: 정답 여부만을 제시하거나 모범답안에 맞춰 평가하는 피드백
- **Label 3**: 제시된 평가 루브릭에 따라 글 전반을 평가하는 피드백
- **Label 4**: 글에서 고쳐야할 부분이나 잘못된 점을 단순히 지적하는 피드백
- **Label 5**: 글에서 잘된 점 혹은 잘못된 점을 구체적으로 언급하고 그에 대한 이유를 제시하며 글에 대한 자신의 생각을 논리적으로 밝히는 피드백
- **Label 6**: 글에서 잘못된 점을 언급한 후 바른 내용으로 수정하는 대안을 제시하거나, 다른 관점/견해/정보를 제시하는 피드백

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# 출력 설정
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 100)

In [2]:
# 레이블 정의
LABEL_NAMES = {
    0: "무관/의미없음",
    1: "막연한 인상",
    2: "정오 판정",
    3: "루브릭 평가",
    4: "오류 지적",
    5: "구체적 설명",
    6: "대안 제시"
}

LABELS = list(range(7))

## 1. 데이터 로드

In [ ]:
# 원본 피드백 데이터 로드 (feedback_text 포함)
feedback_df = pd.read_excel('../data/feedback_data.xlsx')
print(f"원본 피드백 데이터: {len(feedback_df)} rows")
print(f"컬럼: {feedback_df.columns.tolist()}")
feedback_df.head()

In [ ]:
# LLM predictions 파일 로드
LLM_RESULTS_DIR = Path("../4_llm/results/baseline")

pred_files = list(LLM_RESULTS_DIR.glob("*predictions*.csv"))
print(f"발견된 predictions 파일 수: {len(pred_files)}")

all_dfs = []
for f in sorted(pred_files):
    try:
        df = pd.read_csv(f)
        df['source_file'] = f.name
        all_dfs.append(df)
        print(f"  로드됨: {f.name} ({len(df)} rows)")
    except Exception as e:
        print(f"  오류: {f.name} - {e}")

# 통합
df_pred = pd.concat(all_dfs, ignore_index=True)
print(f"\n총 예측 데이터: {len(df_pred)} rows")

In [5]:
# 데이터 전처리
df_pred = df_pred.dropna(subset=['y_true', 'y_pred']).copy()
df_pred['y_true'] = df_pred['y_true'].astype(int)
df_pred['y_pred'] = df_pred['y_pred'].astype(int)

# 오분류 여부 컬럼 추가
df_pred['is_misclassified'] = df_pred['y_true'] != df_pred['y_pred']

print(f"유효 데이터: {len(df_pred)} rows")
print(f"오분류 수: {df_pred['is_misclassified'].sum()} ({df_pred['is_misclassified'].mean()*100:.1f}%)")
print(f"\n모델: {df_pred['model'].unique().tolist()}")
print(f"프롬프트: {sorted(df_pred['prompt_version'].unique().tolist())}")

유효 데이터: 39935 rows
오분류 수: 19109 (47.9%)

모델: ['gpt-4o', 'solar-pro2']
프롬프트: ['P0_v1.0', 'P0_v2.0', 'P1_v1.0', 'P1_v2.0', 'P2_v1.0', 'P2_v3.0', 'P2_v4.0', 'P9_v1.0']


In [6]:
# feedback_text와 조인
df_merged = df_pred.merge(
    feedback_df[['id', 'feedback_text']], 
    left_on='sample_id', 
    right_on='id', 
    how='left'
)

print(f"조인 후 데이터: {len(df_merged)} rows")
print(f"feedback_text 매칭 성공: {df_merged['feedback_text'].notna().sum()} rows")

# 조인 실패 확인
missing = df_merged['feedback_text'].isna().sum()
if missing > 0:
    print(f"\n경고: {missing}개 샘플의 feedback_text를 찾을 수 없음")
    print("매칭 실패 sample_id 예시:", df_merged[df_merged['feedback_text'].isna()]['sample_id'].head().tolist())

조인 후 데이터: 39935 rows
feedback_text 매칭 성공: 39935 rows


## 2. 오분류 통계 요약

In [7]:
# 모델별 오분류율
print("="*60)
print("모델별 오분류율")
print("="*60)
model_stats = df_merged.groupby('model').agg(
    total=('is_misclassified', 'count'),
    misclassified=('is_misclassified', 'sum'),
    error_rate=('is_misclassified', 'mean')
).round(4)
model_stats['error_rate_pct'] = (model_stats['error_rate'] * 100).round(2)
print(model_stats)

모델별 오분류율
            total  misclassified  error_rate  error_rate_pct
model                                                       
gpt-4o       8736           3816      0.4368           43.68
solar-pro2  31199          15293      0.4902           49.02


In [8]:
# 프롬프트별 오분류율
print("="*60)
print("프롬프트별 오분류율")
print("="*60)
prompt_stats = df_merged.groupby('prompt_version').agg(
    total=('is_misclassified', 'count'),
    misclassified=('is_misclassified', 'sum'),
    error_rate=('is_misclassified', 'mean')
).round(4)
prompt_stats['error_rate_pct'] = (prompt_stats['error_rate'] * 100).round(2)
print(prompt_stats.sort_values('error_rate'))

프롬프트별 오분류율
                total  misclassified  error_rate  error_rate_pct
prompt_version                                                  
P2_v3.0          7488           2984      0.3985           39.85
P2_v4.0          2496           1019      0.4083           40.83
P9_v1.0          7487           3146      0.4202           42.02
P1_v2.0          4992           2103      0.4213           42.13
P2_v1.0          4992           2282      0.4571           45.71
P1_v1.0          2496           1376      0.5513           55.13
P0_v1.0          2496           1528      0.6122           61.22
P0_v2.0          7488           4671      0.6238           62.38


In [9]:
# 모델×프롬프트 조합별 오분류율
print("="*60)
print("모델×프롬프트 조합별 오분류율")
print("="*60)
combo_stats = df_merged.groupby(['model', 'prompt_version']).agg(
    total=('is_misclassified', 'count'),
    misclassified=('is_misclassified', 'sum'),
    error_rate=('is_misclassified', 'mean')
).round(4)
combo_stats['error_rate_pct'] = (combo_stats['error_rate'] * 100).round(2)
print(combo_stats.sort_values('error_rate'))

모델×프롬프트 조합별 오분류율
                           total  misclassified  error_rate  error_rate_pct
model      prompt_version                                                  
gpt-4o     P2_v4.0          1248            453      0.3630           36.30
           P2_v3.0          2496            936      0.3750           37.50
           P1_v2.0          1248            489      0.3918           39.18
solar-pro2 P2_v3.0          4992           2048      0.4103           41.03
           P9_v1.0          6239           2608      0.4180           41.80
gpt-4o     P9_v1.0          1248            538      0.4311           43.11
solar-pro2 P1_v2.0          3744           1614      0.4311           43.11
           P2_v4.0          1248            566      0.4535           45.35
           P2_v1.0          4992           2282      0.4571           45.71
           P1_v1.0          2496           1376      0.5513           55.13
gpt-4o     P0_v2.0          2496           1400      0.5609           5

In [10]:
# 레이블별 오분류율
print("="*60)
print("실제 레이블별 오분류율 (어떤 레이블이 분류하기 어려운가?)")
print("="*60)
label_stats = df_merged.groupby('y_true').agg(
    total=('is_misclassified', 'count'),
    misclassified=('is_misclassified', 'sum'),
    error_rate=('is_misclassified', 'mean')
).round(4)
label_stats['error_rate_pct'] = (label_stats['error_rate'] * 100).round(2)
label_stats['label_name'] = label_stats.index.map(LABEL_NAMES)
print(label_stats.sort_values('error_rate', ascending=False))

실제 레이블별 오분류율 (어떤 레이블이 분류하기 어려운가?)
        total  misclassified  error_rate  error_rate_pct label_name
y_true                                                             
5        5280           3796      0.7189           71.89     구체적 설명
3       14016           9522      0.6794           67.94     루브릭 평가
4        3456           1194      0.3455           34.55      오류 지적
6         672            224      0.3333           33.33      대안 제시
2        5472           1606      0.2935           29.35      정오 판정
1        9887           2767      0.2799           27.99     막연한 인상
0        1152              0      0.0000            0.00    무관/의미없음


**결과 해석:**

레이블별 오분류율은 피드백 유형의 **분류 난이도**를 나타냅니다.

- **Label 5(구체적 설명)**: 오분류율 71.9%로 가장 어려움 — "~을 잘 이해했다"처럼 간결하게 표현된 구체적 설명이 막연한 인상이나 루브릭 평가로 오분류됨
- **Label 3(루브릭 평가)**: 오분류율 67.9% — "근거가 부족하다"처럼 간결한 루브릭 평가가 오류 지적과 혼동됨
- **Label 0(무관/의미없음)**: 오분류율 0% — LLM이 가장 잘 식별하는 범주
- **시사점**: 고수준 피드백(Label 3, 5)일수록 언어적 경계가 불명확하여 LLM 분류가 어려움

## 3. 오분류 피드백 텍스트 추출 함수

In [11]:
def extract_misclassified_texts(
    df, 
    model=None, 
    prompt_version=None, 
    true_label=None, 
    pred_label=None,
    limit=None
):
    """
    조건에 맞는 오분류 피드백 텍스트 추출
    
    Args:
        df: 전체 데이터프레임
        model: 특정 모델만 필터링 (None이면 전체)
        prompt_version: 특정 프롬프트만 필터링 (None이면 전체)
        true_label: 실제 레이블 필터링 (None이면 전체)
        pred_label: 예측 레이블 필터링 (None이면 전체)
        limit: 결과 제한 수 (None이면 전체)
    
    Returns:
        오분류 샘플 데이터프레임
    """
    # 오분류만 필터링
    result = df[df['is_misclassified']].copy()
    
    # 조건 필터링
    if model is not None:
        result = result[result['model'] == model]
    if prompt_version is not None:
        result = result[result['prompt_version'] == prompt_version]
    if true_label is not None:
        result = result[result['y_true'] == true_label]
    if pred_label is not None:
        result = result[result['y_pred'] == pred_label]
    
    # 레이블 이름 추가
    result['true_label_name'] = result['y_true'].map(LABEL_NAMES)
    result['pred_label_name'] = result['y_pred'].map(LABEL_NAMES)
    
    # 정렬 (같은 텍스트끼리 모이도록)
    result = result.sort_values(['feedback_text', 'model', 'prompt_version'])
    
    if limit:
        result = result.head(limit)
    
    return result[[
        'sample_id', 'model', 'prompt_version', 
        'y_true', 'true_label_name', 
        'y_pred', 'pred_label_name', 
        'feedback_text'
    ]]

In [12]:
def get_consistently_misclassified_texts(df, min_error_count=3):
    """
    여러 모델/프롬프트에서 일관되게 오분류되는 텍스트 추출
    
    Args:
        df: 전체 데이터프레임
        min_error_count: 최소 오분류 횟수 (여러 조합에서 오분류된 경우)
    
    Returns:
        일관되게 오분류되는 샘플 데이터프레임
    """
    # 샘플별 오분류 횟수 계산
    misclassified = df[df['is_misclassified']]
    error_counts = misclassified.groupby('sample_id').size().reset_index(name='error_count')
    
    # 최소 오분류 횟수 이상인 샘플
    consistent_errors = error_counts[error_counts['error_count'] >= min_error_count]
    
    # 해당 샘플의 상세 정보
    result = df[df['sample_id'].isin(consistent_errors['sample_id'])].copy()
    result = result.merge(error_counts, on='sample_id')
    
    # 레이블 이름 추가
    result['true_label_name'] = result['y_true'].map(LABEL_NAMES)
    result['pred_label_name'] = result['y_pred'].map(LABEL_NAMES)
    
    return result.sort_values('error_count', ascending=False)

In [13]:
def get_confusion_pair_texts(df, true_label, pred_label, limit=20):
    """
    특정 혼동 패턴(true→pred)의 피드백 텍스트 추출
    
    Args:
        df: 전체 데이터프레임
        true_label: 실제 레이블
        pred_label: 예측 레이블
        limit: 결과 제한 수
    
    Returns:
        해당 혼동 패턴의 샘플 데이터프레임
    """
    result = df[
        (df['y_true'] == true_label) & 
        (df['y_pred'] == pred_label)
    ].copy()
    
    # 유니크한 텍스트만 추출
    unique_texts = result.drop_duplicates(subset=['feedback_text'])
    
    unique_texts['true_label_name'] = unique_texts['y_true'].map(LABEL_NAMES)
    unique_texts['pred_label_name'] = unique_texts['y_pred'].map(LABEL_NAMES)
    
    return unique_texts[[
        'sample_id', 'y_true', 'true_label_name', 
        'y_pred', 'pred_label_name', 'feedback_text'
    ]].head(limit)

## 4. 모델별 오분류 텍스트 분석

In [14]:
# 모델 목록
models = df_merged['model'].unique().tolist()
print(f"분석할 모델: {models}")

분석할 모델: ['gpt-4o', 'solar-pro2']


In [15]:
# 모델별 오분류 분석
for model in models:
    print("\n" + "="*70)
    print(f"[{model}] 오분류 분석")
    print("="*70)
    
    model_df = df_merged[df_merged['model'] == model]
    model_misclassified = model_df[model_df['is_misclassified']]
    
    print(f"총 예측: {len(model_df)}, 오분류: {len(model_misclassified)} ({len(model_misclassified)/len(model_df)*100:.1f}%)")
    
    # 가장 많은 오분류 패턴
    confusion_patterns = model_misclassified.groupby(['y_true', 'y_pred']).size().reset_index(name='count')
    confusion_patterns = confusion_patterns.sort_values('count', ascending=False)
    confusion_patterns['pattern'] = confusion_patterns.apply(
        lambda r: f"{LABEL_NAMES[r['y_true']]} → {LABEL_NAMES[r['y_pred']]}", axis=1
    )
    
    print(f"\n상위 5개 오분류 패턴:")
    for _, row in confusion_patterns.head(5).iterrows():
        print(f"  {row['pattern']}: {row['count']}건")


[gpt-4o] 오분류 분석
총 예측: 8736, 오분류: 3816 (43.7%)

상위 5개 오분류 패턴:
  루브릭 평가 → 오류 지적: 1009건
  루브릭 평가 → 구체적 설명: 462건
  루브릭 평가 → 정오 판정: 347건
  루브릭 평가 → 막연한 인상: 306건
  막연한 인상 → 무관/의미없음: 303건

[solar-pro2] 오분류 분석
총 예측: 31199, 오분류: 15293 (49.0%)

상위 5개 오분류 패턴:
  루브릭 평가 → 정오 판정: 2208건
  루브릭 평가 → 오류 지적: 2180건
  루브릭 평가 → 막연한 인상: 2106건
  막연한 인상 → 무관/의미없음: 1370건
  구체적 설명 → 오류 지적: 1014건


In [16]:
# 첫 번째 모델의 오분류 텍스트 예시 (상위 20개)
model = models[0]
print(f"\n[{model}] 오분류 텍스트 예시 (상위 20개)")
print("-"*70)

misclassified_texts = extract_misclassified_texts(df_merged, model=model, limit=20)
for idx, row in misclassified_texts.iterrows():
    print(f"\n[{row['sample_id']}]")
    print(f"  정답: {row['y_true']} ({row['true_label_name']})")
    print(f"  예측: {row['y_pred']} ({row['pred_label_name']})")
    print(f"  텍스트: {row['feedback_text'][:100]}..." if len(str(row['feedback_text'])) > 100 else f"  텍스트: {row['feedback_text']}")

## 5. 프롬프트별 오분류 텍스트 분석

In [17]:
# 프롬프트 목록
prompts = sorted(df_merged['prompt_version'].unique().tolist())
print(f"분석할 프롬프트: {prompts}")

분석할 프롬프트: ['P0_v1.0', 'P0_v2.0', 'P1_v1.0', 'P1_v2.0', 'P2_v1.0', 'P2_v3.0', 'P2_v4.0', 'P9_v1.0']


In [18]:
# 프롬프트별 오분류 분석
for prompt in prompts:
    print("\n" + "="*70)
    print(f"[{prompt}] 오분류 분석")
    print("="*70)
    
    prompt_df = df_merged[df_merged['prompt_version'] == prompt]
    prompt_misclassified = prompt_df[prompt_df['is_misclassified']]
    
    print(f"총 예측: {len(prompt_df)}, 오분류: {len(prompt_misclassified)} ({len(prompt_misclassified)/len(prompt_df)*100:.1f}%)")
    
    # 가장 많은 오분류 패턴
    confusion_patterns = prompt_misclassified.groupby(['y_true', 'y_pred']).size().reset_index(name='count')
    confusion_patterns = confusion_patterns.sort_values('count', ascending=False)
    confusion_patterns['pattern'] = confusion_patterns.apply(
        lambda r: f"{LABEL_NAMES[r['y_true']]} → {LABEL_NAMES[r['y_pred']]}", axis=1
    )
    
    print(f"\n상위 5개 오분류 패턴:")
    for _, row in confusion_patterns.head(5).iterrows():
        print(f"  {row['pattern']}: {row['count']}건")


[P0_v1.0] 오분류 분석
총 예측: 2496, 오분류: 1528 (61.2%)

상위 5개 오분류 패턴:
  루브릭 평가 → 정오 판정: 270건
  루브릭 평가 → 막연한 인상: 252건
  막연한 인상 → 무관/의미없음: 160건
  루브릭 평가 → 오류 지적: 142건
  막연한 인상 → 정오 판정: 122건

[P0_v2.0] 오분류 분석
총 예측: 7488, 오분류: 4671 (62.4%)

상위 5개 오분류 패턴:
  루브릭 평가 → 정오 판정: 838건
  루브릭 평가 → 오류 지적: 830건
  루브릭 평가 → 막연한 인상: 604건
  막연한 인상 → 무관/의미없음: 582건
  구체적 설명 → 정오 판정: 257건

[P1_v1.0] 오분류 분석
총 예측: 2496, 오분류: 1376 (55.1%)

상위 5개 오분류 패턴:
  루브릭 평가 → 막연한 인상: 276건
  루브릭 평가 → 정오 판정: 248건
  막연한 인상 → 무관/의미없음: 122건
  구체적 설명 → 막연한 인상: 108건
  구체적 설명 → 루브릭 평가: 106건

[P1_v2.0] 오분류 분석
총 예측: 4992, 오분류: 2103 (42.1%)

상위 5개 오분류 패턴:
  루브릭 평가 → 오류 지적: 445건
  루브릭 평가 → 구체적 설명: 242건
  루브릭 평가 → 정오 판정: 236건
  루브릭 평가 → 막연한 인상: 198건
  구체적 설명 → 오류 지적: 154건

[P2_v1.0] 오분류 분석
총 예측: 4992, 오분류: 2282 (45.7%)

상위 5개 오분류 패턴:
  루브릭 평가 → 오류 지적: 312건
  루브릭 평가 → 막연한 인상: 280건
  루브릭 평가 → 정오 판정: 252건
  막연한 인상 → 무관/의미없음: 250건
  구체적 설명 → 루브릭 평가: 206건

[P2_v3.0] 오분류 분석
총 예측: 7488, 오분류: 2984 (39.9%)

상위 5개 오분류 패턴:
  루브릭 평가 → 오류 지적: 748건
  구체적 설

## 6. 특정 혼동 패턴 심층 분석

In [19]:
# 전체에서 가장 빈번한 혼동 패턴 계산
all_misclassified = df_merged[df_merged['is_misclassified']]
top_patterns = all_misclassified.groupby(['y_true', 'y_pred']).size().reset_index(name='count')
top_patterns = top_patterns.sort_values('count', ascending=False)
top_patterns['pattern'] = top_patterns.apply(
    lambda r: f"{LABEL_NAMES[r['y_true']]} → {LABEL_NAMES[r['y_pred']]}", axis=1
)

print("가장 빈번한 혼동 패턴 Top 10:")
print(top_patterns.head(10).to_string(index=False))

가장 빈번한 혼동 패턴 Top 10:
 y_true  y_pred  count          pattern
      3       4   3189   루브릭 평가 → 오류 지적
      3       2   2555   루브릭 평가 → 정오 판정
      3       1   2412  루브릭 평가 → 막연한 인상
      1       0   1673 막연한 인상 → 무관/의미없음
      5       3   1310  구체적 설명 → 루브릭 평가
      3       5   1193  루브릭 평가 → 구체적 설명
      5       4   1158   구체적 설명 → 오류 지적
      5       1    843  구체적 설명 → 막연한 인상
      2       1    671   정오 판정 → 막연한 인상
      1       2    596   막연한 인상 → 정오 판정


In [20]:
# 상위 3개 혼동 패턴에 대한 텍스트 예시
for _, pattern in top_patterns.head(3).iterrows():
    true_l = pattern['y_true']
    pred_l = pattern['y_pred']
    
    print("\n" + "="*70)
    print(f"혼동 패턴: {LABEL_NAMES[true_l]} (정답) → {LABEL_NAMES[pred_l]} (예측)")
    print(f"총 {pattern['count']}건 발생")
    print("="*70)
    
    texts = get_confusion_pair_texts(df_merged, true_l, pred_l, limit=10)
    
    print(f"\n유니크 텍스트 예시:")
    for idx, (_, row) in enumerate(texts.iterrows(), 1):
        text = str(row['feedback_text'])
        print(f"\n{idx}. [{row['sample_id']}]")
        print(f"   {text[:150]}..." if len(text) > 150 else f"   {text}")

## 7. 일관되게 오분류되는 텍스트 (난이도가 높은 샘플)

In [21]:
# 여러 모델/프롬프트에서 일관되게 오분류되는 텍스트
# (모든 조합에서 오분류 = 가장 분류하기 어려운 샘플)

n_combinations = df_merged.groupby(['model', 'prompt_version']).ngroups
print(f"총 모델×프롬프트 조합 수: {n_combinations}")

# 샘플별 오분류 횟수
sample_error_counts = df_merged.groupby('sample_id').agg(
    total_predictions=('is_misclassified', 'count'),
    error_count=('is_misclassified', 'sum'),
    error_rate=('is_misclassified', 'mean')
).reset_index()

# 오분류가 많은 순으로 정렬
sample_error_counts = sample_error_counts.sort_values('error_count', ascending=False)

print(f"\n오분류가 가장 많은 샘플 Top 20:")
print(sample_error_counts.head(20).to_string(index=False))

In [22]:
# 오분류율이 50% 이상인 샘플의 텍스트 확인
high_error_samples = sample_error_counts[sample_error_counts['error_rate'] >= 0.5]
print(f"\n오분류율 50% 이상 샘플 수: {len(high_error_samples)}")

# 해당 샘플의 상세 정보
high_error_details = df_merged[df_merged['sample_id'].isin(high_error_samples['sample_id'])]
high_error_unique = high_error_details.drop_duplicates(subset=['sample_id'])

print(f"\n[오분류율 50% 이상] 피드백 텍스트 예시:")
print("-"*70)

for _, row in high_error_unique.head(15).iterrows():
    sample_id = row['sample_id']
    text = str(row['feedback_text'])
    true_label = row['y_true']
    
    # 이 샘플의 예측 분포
    sample_preds = high_error_details[high_error_details['sample_id'] == sample_id]['y_pred'].value_counts().to_dict()
    pred_dist = ', '.join([f"{LABEL_NAMES.get(k, k)}:{v}" for k, v in sorted(sample_preds.items())])
    
    print(f"\n[{sample_id}]")
    print(f"  정답: {true_label} ({LABEL_NAMES[true_label]})")
    print(f"  예측 분포: {pred_dist}")
    print(f"  텍스트: {text[:120]}..." if len(text) > 120 else f"  텍스트: {text}")

## 8. 모델×프롬프트 조합별 고유 오분류 (특정 조합에서만 틀리는 경우)

In [23]:
def get_unique_errors_by_combination(df):
    """
    특정 모델×프롬프트 조합에서만 오분류되는 샘플 찾기
    (다른 조합에서는 정분류되는 샘플)
    """
    results = []
    
    combinations = df.groupby(['model', 'prompt_version']).groups.keys()
    
    for combo in combinations:
        model, prompt = combo
        
        # 이 조합에서 오분류된 샘플
        combo_df = df[(df['model'] == model) & (df['prompt_version'] == prompt)]
        combo_errors = set(combo_df[combo_df['is_misclassified']]['sample_id'])
        
        # 다른 조합들에서 오분류된 샘플
        other_df = df[(df['model'] != model) | (df['prompt_version'] != prompt)]
        other_errors = set(other_df[other_df['is_misclassified']]['sample_id'])
        
        # 이 조합에서만 오분류된 샘플 (다른 곳에서는 모두 정분류)
        unique_errors = combo_errors - other_errors
        
        if unique_errors:
            results.append({
                'model': model,
                'prompt_version': prompt,
                'unique_error_count': len(unique_errors),
                'sample_ids': list(unique_errors)
            })
    
    return pd.DataFrame(results).sort_values('unique_error_count', ascending=False)

In [24]:
# 조합별 고유 오분류 분석
unique_errors_df = get_unique_errors_by_combination(df_merged)

print("모델×프롬프트 조합별 고유 오분류 수:")
print("(해당 조합에서만 틀리고 다른 조합에서는 맞춘 샘플)")
print("="*60)

if len(unique_errors_df) > 0:
    for _, row in unique_errors_df.iterrows():
        print(f"\n[{row['model']} + {row['prompt_version']}]")
        print(f"  고유 오분류 수: {row['unique_error_count']}")
        
        # 해당 샘플들의 텍스트 예시
        sample_ids = row['sample_ids'][:3]  # 최대 3개만
        for sid in sample_ids:
            sample_data = df_merged[
                (df_merged['sample_id'] == sid) & 
                (df_merged['model'] == row['model']) &
                (df_merged['prompt_version'] == row['prompt_version'])
            ].iloc[0]
            print(f"\n  [{sid}]")
            print(f"    정답: {sample_data['y_true']} ({LABEL_NAMES[sample_data['y_true']]})")
            print(f"    예측: {sample_data['y_pred']} ({LABEL_NAMES[sample_data['y_pred']]})")
            text = str(sample_data['feedback_text'])
            print(f"    텍스트: {text[:100]}..." if len(text) > 100 else f"    텍스트: {text}")
else:
    print("모든 조합에서 공통으로 오분류되는 패턴만 존재합니다.")

## 9. 결과 저장

In [25]:
# 결과 저장 디렉토리
OUTPUT_DIR = Path('results/misclassified_texts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"결과 저장 경로: {OUTPUT_DIR}")

결과 저장 경로: results/misclassified_texts


In [26]:
# 1. 전체 오분류 데이터 저장
all_misclassified_with_text = df_merged[df_merged['is_misclassified']][[
    'sample_id', 'model', 'prompt_version', 
    'y_true', 'y_pred', 'feedback_text'
]].copy()
all_misclassified_with_text['true_label_name'] = all_misclassified_with_text['y_true'].map(LABEL_NAMES)
all_misclassified_with_text['pred_label_name'] = all_misclassified_with_text['y_pred'].map(LABEL_NAMES)

all_misclassified_with_text.to_csv(OUTPUT_DIR / 'all_misclassified.csv', index=False, encoding='utf-8-sig')
print(f"저장됨: all_misclassified.csv ({len(all_misclassified_with_text)} rows)")

저장됨: all_misclassified.csv (19109 rows)


In [27]:
# 2. 모델별 오분류 데이터 저장
for model in models:
    model_data = all_misclassified_with_text[all_misclassified_with_text['model'] == model]
    filename = f"misclassified_{model.replace('/', '_')}.csv"
    model_data.to_csv(OUTPUT_DIR / filename, index=False, encoding='utf-8-sig')
    print(f"저장됨: {filename} ({len(model_data)} rows)")

저장됨: misclassified_gpt-4o.csv (3816 rows)
저장됨: misclassified_solar-pro2.csv (15293 rows)


In [28]:
# 3. 고난이도 샘플 (오분류율 50% 이상) 저장
high_error_samples_df = sample_error_counts[sample_error_counts['error_rate'] >= 0.5].copy()
high_error_samples_df = high_error_samples_df.merge(
    feedback_df[['id', 'feedback_text', 'label']], 
    left_on='sample_id', 
    right_on='id'
)
high_error_samples_df['label_name'] = high_error_samples_df['label'].map(LABEL_NAMES)
high_error_samples_df.to_csv(OUTPUT_DIR / 'high_error_rate_samples.csv', index=False, encoding='utf-8-sig')
print(f"저장됨: high_error_rate_samples.csv ({len(high_error_samples_df)} rows)")

저장됨: high_error_rate_samples.csv (190 rows)


In [29]:
# 4. 혼동 패턴별 텍스트 예시 저장
confusion_examples = []
for _, pattern in top_patterns.head(10).iterrows():
    true_l = pattern['y_true']
    pred_l = pattern['y_pred']
    
    texts = get_confusion_pair_texts(df_merged, true_l, pred_l, limit=20)
    texts['confusion_pattern'] = f"{true_l}→{pred_l}"
    texts['pattern_description'] = f"{LABEL_NAMES[true_l]} → {LABEL_NAMES[pred_l]}"
    confusion_examples.append(texts)

confusion_examples_df = pd.concat(confusion_examples, ignore_index=True)
confusion_examples_df.to_csv(OUTPUT_DIR / 'confusion_pattern_examples.csv', index=False, encoding='utf-8-sig')
print(f"저장됨: confusion_pattern_examples.csv ({len(confusion_examples_df)} rows)")

저장됨: confusion_pattern_examples.csv (199 rows)


In [30]:
# 저장된 파일 목록
print("\n" + "="*60)
print("저장된 파일 목록")
print("="*60)
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    size = f.stat().st_size / 1024
    print(f"  {f.name} ({size:.1f} KB)")


저장된 파일 목록
  all_misclassified.csv (2020.2 KB)
  confusion_pattern_examples.csv (26.4 KB)
  high_error_rate_samples.csv (18.9 KB)
  misclassified_gpt-4o.csv (384.0 KB)
  misclassified_solar-pro2.csv (1636.3 KB)


## 10. 요약

In [31]:
print("="*70)
print("LLM 오분류 피드백 텍스트 추출 요약")
print("="*70)

print(f"\n[데이터 개요]")
print(f"  - 총 예측 수: {len(df_merged):,}")
print(f"  - 오분류 수: {df_merged['is_misclassified'].sum():,} ({df_merged['is_misclassified'].mean()*100:.1f}%)")
print(f"  - 모델 수: {len(models)}")
print(f"  - 프롬프트 수: {len(prompts)}")

print(f"\n[분류하기 어려운 레이블 (오분류율 높은 순)]")
for _, row in label_stats.sort_values('error_rate', ascending=False).head(3).iterrows():
    print(f"  - Label {row.name} ({row['label_name']}): 오분류율 {row['error_rate_pct']}%")

print(f"\n[가장 빈번한 혼동 패턴]")
for _, row in top_patterns.head(5).iterrows():
    print(f"  - {row['pattern']}: {row['count']:,}건")

print(f"\n[고난이도 샘플]")
print(f"  - 오분류율 50% 이상 샘플 수: {len(high_error_samples)}")
print(f"  - 오분류율 80% 이상 샘플 수: {len(sample_error_counts[sample_error_counts['error_rate'] >= 0.8])}")

print(f"\n[저장 위치]")
print(f"  {OUTPUT_DIR}")

LLM 오분류 피드백 텍스트 추출 요약

[데이터 개요]
  - 총 예측 수: 39,935
  - 오분류 수: 19,109 (47.9%)
  - 모델 수: 2
  - 프롬프트 수: 8

[분류하기 어려운 레이블 (오분류율 높은 순)]
  - Label 5 (구체적 설명): 오분류율 71.89%
  - Label 3 (루브릭 평가): 오분류율 67.94%
  - Label 4 (오류 지적): 오분류율 34.55%

[가장 빈번한 혼동 패턴]
  - 루브릭 평가 → 오류 지적: 3,189건
  - 루브릭 평가 → 정오 판정: 2,555건
  - 루브릭 평가 → 막연한 인상: 2,412건
  - 막연한 인상 → 무관/의미없음: 1,673건
  - 구체적 설명 → 루브릭 평가: 1,310건

[고난이도 샘플]
  - 오분류율 50% 이상 샘플 수: 190
  - 오분류율 80% 이상 샘플 수: 118

[저장 위치]
  results/misclassified_texts
